# VarChat — Güçlü yargıç (`qwen2.5:32b`): ham 14B + dört modelin karşılaştırması

1. **Ham `qwen2.5:14b`** aynı 56 girdi için cevap üretir (eğitim yok, Ollama'nın hazır modeli).
2. **`qwen2.5:32b`** dört modelin cevaplarını aynı ölçüyle puanlar: ham `qwen2.5:7b`, `qwen-taban`, `varchat`, ham `qwen2.5:14b`.
   Yargıç, yarışan bütün modellerden büyüktür; kimse kendi cevabını puanlamaz.

Eğitim verisi denetimi bu defterde YAPILMAZ (Colab kredisi harcamasın diye; denetim yerelde yapılıyor).
7B yargıcın eski sonuçları silinmez; yeni sonuçlar `__yargic-qwen2.5-32b` ekli ayrı dosyalara yazılır.

**Kullanım:** *Runtime → Change runtime type → **A100*** (32B yargıç ~22 GB GPU belleği ister; L4'te sığmayabilir, yavaşlar)
→ *Runtime → **Run all***. Drive iznini onayla. Süre: A100'de tahminen 1,5-2 saat. Oturum koparsa yeniden *Run all*:
biten işler tekrarlanmaz. Tarayıcı sekmesini açık tut.

## 0) GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
vram = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()[0])
print(f"GPU belleği: {vram / 1024:.0f} GB")
if vram < 30000:
    print("UYARI: 32B yargıç için A100 (40 GB) önerilir; bu kartta model kısmen işlemciye taşar ve çok yavaşlar.")

## 1) Projeyi getir + paketler

In [ ]:
import os
if not os.path.exists('/content/varchat'):
    !git clone -q https://github.com/atu-ce/varchat.git /content/varchat
else:
    !cd /content/varchat && git fetch -q origin && git checkout -q origin/main -- . ':(exclude)degerlendirme/sonuclar'
%cd /content/varchat
!git log --oneline -1 origin/main
!pip install -q "ollama==0.6.2" "rapidfuzz==3.14.5"

## 2) Drive'a bağlan (cevaplar `colab_olcum` defterinden)

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
SONUC = '/content/drive/MyDrive/varchat-olcum/sonuclar'
assert os.path.exists(SONUC), "Drive'da varchat-olcum/sonuclar yok: önce colab_olcum.ipynb çalışmış olmalı"
yerel = '/content/varchat/degerlendirme/sonuclar'
if not os.path.islink(yerel):
    for ad in os.listdir(yerel):
        kaynak = os.path.join(yerel, ad)
        if os.path.isfile(kaynak) and not os.path.exists(os.path.join(SONUC, ad)):
            shutil.copy2(kaynak, SONUC)
    shutil.rmtree(yerel)
    os.symlink(SONUC, yerel)
uretimler = sorted(a for a in os.listdir(SONUC) if a.startswith("uretim_"))
print("bulunan üretim dosyaları:", uretimler)
assert any("varchat" in a for a in uretimler) and any("qwen-taban" in a for a in uretimler), "qwen-taban / varchat cevapları eksik"

## 3) Ollama + yargıç modeli (`qwen2.5:32b`, ~20 GB indirme)

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests, json, datetime

def ollama_hazir():
    try:
        return requests.get('http://127.0.0.1:11434/api/version', timeout=2).json()
    except Exception:
        return None

if not ollama_hazir():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_hazir():
            break
assert ollama_hazir(), "Ollama başlamadı; /content/ollama.log dosyasına bak"
!ollama pull qwen2.5:32b
!ollama run qwen2.5:32b "Merhaba. Tek kelimeyle cevap ver."
ps_cikti = !ollama ps
print("\n".join(ps_cikti))
if not any("100% GPU" in s for s in ps_cikti[1:]):
    print("UYARI: yargıç tamamen GPU'da değil (yavaş çalışır). A100 seçili mi?")

## 4) Ham `qwen2.5:14b` cevap üretir (aynı 56 girdi, aynı istem ve ayarlar)

In [ ]:
!ollama pull qwen2.5:14b
!python -u degerlendirme/b06_toplu_uret.py qwen2.5:14b

## 5) `qwen2.5:32b` dört modeli puanlar

Asıl karşılaştırmalar: **qwen-taban → varchat** (eğitimin etkisi) ve **varchat → qwen2.5:14b** (eğitilmiş 7B mi, ham 14B mi?).
Ham `qwen2.5:7b` en sonda (zaman yoksa atlanabilir).

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen-taban --yargic qwen2.5:32b

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model varchat --yargic qwen2.5:32b

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen2.5:14b --yargic qwen2.5:32b

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen2.5:7b --yargic qwen2.5:32b

## 6) Sonuçlar

- **Sadakat farkları** (%95 güven aralığıyla): qwen-taban → varchat ve varchat → qwen2.5:14b
- **Yargıç uyumu:** 7B yargıç ile qwen2.5:32b aynı cümlelerde ne sıklıkla aynı kararı verdi (kappa 0.6 üstü iyi, 0.4 altı zayıf)

In [ ]:
!python degerlendirme/b08_karsilastir.py --yargic qwen2.5:32b --ciftler "qwen-taban>varchat,varchat>qwen2.5:14b" --uyum qwen2.5:32b

## 7) Sonuçları bilgisayarına al

Drive'daki **`varchat-olcum/sonuclar`** klasöründen şunları indirip projede `degerlendirme/colab_son/` içine koy:
- adında `__yargic-qwen2.5-32b` geçen bütün dosyalar
- `uretim_qwen2.5-14b_*.jsonl`
- `karsilastirma_istatistik_*`